# WarpKriging with Box-Cox warping (Octave)

The **Box-Cox** warp applies a power transformation $w(x) = (x^\lambda - 1)/\lambda$.
This is useful when the input-output relationship benefits from a monotone nonlinear rescaling.

Steps:
1. Setup mlibkriging / jlibkriging
2. Define the Branin function and plot it
3. Build a space-filling design and evaluate it
4. Fit a `WarpKriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters

## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).
It has three global minima.

In [2]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

% 50x50 evaluation grid
grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmps_n225uu/plots/tmpfuvurl89 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 2. Design of experiments

We sample $n = 30$ points using a Latin Hypercube Design.

In [3]:
rand('seed', 42);
n = 30; d = 2;

% Simple LHS: stratified uniform sample, independently permuted per dimension
X = zeros(n, d);
for j = 1:d
    perm = randperm(n);
    X(:,j) = (perm' - rand(n,1)) / n;
end
y = branin(X);

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title(sprintf('%d LHS design points on Branin', n));
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmps_n225uu/plots/tmp4rztwig9 does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 3. Fit a WarpKriging model (`boxcox`)

We use `warping = {'boxcox', 'boxcox'}` — one warp per input dimension.

In [4]:
wk = WarpKriging(y, X, {'boxcox', 'boxcox'}, 'matern5_2', 'constant', false, 'BFGS+Adam', 'LL');
disp(wk.summary())

* WarpKriging
  - kernel:      matern5_2
  - regmodel:    constant
  - normalize:   false
  - n obs:       30
  - d input:     2
  - d features:  2
  - warpings:
      x0: "boxcox"  →  BoxCox(lambda=1)
      x1: "boxcox"  →  BoxCox(lambda=1)
  - sigma2:      1.62784e+06
  - theta:          1.4390   4.4902
  - beta:           1.3573e+03
  - LL:          -100.09
  - total warp params: 2



## 4. Predict on a fine grid

`predict()` returns the posterior mean and standard deviation at new points.

In [5]:
[p_mean, p_stdev] = wk.predict(grid_pts, true, false);
z_mean = reshape(p_mean, 50, 50);
z_sd   = reshape(p_stdev, 50, 50);

vmin = min(min(z_true(:)), min(z_mean(:)));
vmax = max(max(z_true(:)), max(z_mean(:)));

figure;
subplot(1, 2, 1);
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('True Branin'); xlabel('x_1'); ylabel('x_2');

subplot(1, 2, 2);
contourf(G1, G2, z_mean, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('WarpKriging (boxcox) mean'); xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmps_n225uu/plots/tmp_p1wo29q does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



In [6]:
% Posterior standard deviation (uncertainty)
figure;
contourf(G1, G2, z_sd, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title('WarpKriging (boxcox) std dev (uncertainty)');
xlabel('x_1'); ylabel('x_2');

Inline plot failed, consider trying another graphics toolkit
error: print: directory /tmp/tmps_n225uu/plots/tmpct_xl4vq does not exist
error: called from
    _make_figures>safe_print at line 125 column 7
    _make_figures at line 49 column 13



## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood, and warping specification.

In [7]:
fprintf('Kernel       : %s\n', wk.kernel());
fprintf('Theta (range): %s\n', mat2str(wk.theta(), 4));
fprintf('Sigma2       : %.4f\n', wk.sigma2());
fprintf('LogLikelihood: %.4f\n', wk.logLikelihood());
fprintf('Feature dim  : %d\n', wk.feature_dim());
fprintf('Warping      : ');
w = wk.warping(); for i = 1:length(w); fprintf('%s ', w{i}); end; fprintf('\n');

Kernel       : matern5_2


Theta (range): [1.439;4.49]


Sigma2       : 1627837.0512


LogLikelihood: -100.0902


Feature dim  : 2


Warping      : 

boxcox boxcox 
